# RAG Pipeline for MCQ Solver  — Kaggle Edition

## What is RAG?
**Retrieval-Augmented Generation (RAG)** enhances answer prediction by first *retrieving* relevant
external documents (our Wikipedia corpus) and then using that retrieved context to *rank* the MCQ options.

## Kaggle Setup Instructions

Before running this notebook you need to attach **two datasets** from the Kaggle sidebar → *Data* → *+ Add Data*:

| What | Kaggle Dataset slug | Path inside notebook |
|------|--------------------|----------------------|
| Competition data (train/test CSVs) | the competition dataset | `/kaggle/input/<competition-slug>/` |
| Wikipedia corpus (.md files) | your uploaded wiki corpus dataset | `/kaggle/input/<your-wiki-dataset-slug>/wiki_corpus/` |

> **How to upload the wiki corpus to Kaggle:**  
> Kaggle Datasets → New Dataset → upload the `wiki_corpus/` folder as a zip → use the slug here.

Edit the two `*_SLUG` constants in the **Setup** cell below to match your actual Kaggle dataset slugs.

## Pipeline Overview
```
OFFLINE (done once per session)
  1. Load all .md files from the wiki corpus dataset
  2. Chunk each article into overlapping 500-char passages
  3. Fit a TF-IDF vectorizer  →  sparse (chunks x vocab) matrix

ONLINE (per question)
  4. Embed the question prompt  →  TF-IDF query vector
  5. Retrieve top-K most similar chunks (cosine similarity)
  6. Score each option: cosine_sim(context, prompt + option)
  7. Rank options → top-3 predictions  →  submission CSV
```

## Notebook Sections
1. Environment Detection and Path Setup
2. Install / Import Dependencies
3. Load and Explore the MCQ Dataset
4. Load and Chunk the Wikipedia Corpus
5. Build the TF-IDF Retrieval Index
6. Define the Retrieval Function
7. Define the Scoring and Ranking Function
8. Evaluate on Training Set (MAP@3)
9. Generate and Save the Submission File
10. Bonus — Dense Retrieval with Sentence-Transformers (GPU)

In [108]:
!mkdir -p ~/.kaggle && echo KGAT_bbdd41cf9a90831eae65f89984ef77e8 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [88]:
import os
import re
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

In [89]:

_this_dir       = os.path.dirname(os.path.abspath("__file__"))
COMPETITION_DIR = _this_dir
CORPUS_DIR      = os.path.join(_this_dir, "wiki_corpus")
OUTPUT_DIR      = _this_dir

TRAIN_PATH  = os.path.join(COMPETITION_DIR, "train.csv")
TEST_PATH   = os.path.join(COMPETITION_DIR, "test.csv")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

In [90]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train rows: {len(train_df):,}  |  Columns: {list(train_df.columns)}")
print(f"Test  rows: {len(test_df):,}   |  Columns: {list(test_df.columns)}")

Train rows: 2,000  |  Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test  rows: 500   |  Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']


## Load and Chunk the Wikipedia Corpus

In [91]:
# !pip install chonkie

In [ ]:
from chonkie import RecursiveChunker
# from chonkie import SemanticChunker

TAIL_SECTIONS = (
    "See also", "References", "Further reading",
    "External links", "Notes", "Bibliography", "Sources",
)
TAIL_PATTERN = re.compile(
    r"^==\s*(?:" + "|".join(TAIL_SECTIONS) + r")\s*==\s*$",
    flags=re.MULTILINE | re.IGNORECASE,
)


def drop_tail_sections(text: str) -> str:
    # Cut at the earliest tail heading; everything below it is boilerplate.
    match = TAIL_PATTERN.search(text)
    return text[: match.start()] if match else text


def load_and_chunk_corpus(
    corpus_dir: str,
    chunk_size: int = 500,
) -> list:
    # Returns: List of dicts, each with keys 'text' (str) and 'source' (str).

    
    chunker = RecursiveChunker(chunk_size=chunk_size)

    chunks = []
    chars_before = chars_after = 0
    md_files = glob.glob(os.path.join(corpus_dir, "*.md"))
    print(f"Found {len(md_files)} .md files in corpus.")
    for filepath in tqdm(md_files, desc="Chunking corpus"):
        filename = os.path.basename(filepath)
        with open(filepath, "r", encoding="utf-8") as f:
            raw = f.read()
        # ── Clean the Markdown 
        chars_before += len(raw)
        text = drop_tail_sections(raw)
        chars_after += len(text)
        # Strip heading markers (## Introduction) but keep the heading text
        text = re.sub(r"^#{1,6}\s+", "", text, flags=re.MULTILINE)
        # Drop HTML/XML tags (e.g. <ref>, <math>)
        text = re.sub(r"<[^>]+>", "", text)
        # Drop front-matter blockquote lines added by our fetcher
        # (lines starting with "> **Query Topic**:", "> **Wikipedia Page**:", etc.)
        text = re.sub(r"^>.*$", "", text, flags=re.MULTILINE)
        # Collapse runs of blank lines
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if not text:
            continue  # skip empty or near-empty files
        
        # ── Chunking via chonkie
        for c in chunker.chunk(text):
            if len(c.text.strip()) > 50:
                chunks.append({"text": c.text, "source": filename})

    removed = chars_before - chars_after
    print(f"Tail sections removed: {removed:,} chars "
          f"({removed / max(chars_before, 1):.1%} of the raw corpus)")
    return chunks

all_chunks = load_and_chunk_corpus(CORPUS_DIR, chunk_size=500)
chunk_texts = [c["text"] for c in all_chunks]   # plain text list for TF-IDF
print(f"\nTotal chunks: {len(all_chunks):,}")

# Confirm the boilerplate is actually gone from the index.
_leftover = sum(1 for t in chunk_texts if TAIL_PATTERN.search(t))
print(f"Chunks still containing a tail heading: {_leftover}")



# Start a new wandb run to track this script.
# run = wandb.init(
#     entity="vivekkumarkamal-indian-institute-of-technology-madras",
#     name=""
#     project="21f3002412-t22026",
#     config={
#         "chunking_method": "chonkie-recursive",
#         "chunk_size": 500,
#         "num_chunks": len(all_chunks),
#     },
# )

# # Log the actual chunks as a table so you can inspect them in the wandb UI
# chunk_table = wandb.Table(columns=["source", "text"])
# for c in all_chunks:
#     chunk_table.add_data(c["source"], c["text"])
# run.log({"chunks": chunk_table})

# run.finish()

Found 508 .md files in corpus.


Chunking corpus: 100%|██████████| 508/508 [00:00<00:00, 768.25it/s]


Tail sections removed: 510,047 chars (3.5% of the raw corpus)

Total chunks: 38,891
Chunks still containing a tail heading: 0


In [93]:
vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    min_df=2,            # drop terms appearing in fewer than 2 chunks
    max_df=0.95,         # drop terms in 95%+ of chunks (so common they're noise)
    ngram_range=(1, 2),  # include unigrams and bigrams
    stop_words="english",
)

print("Fitting TF-IDF vectorizer on all chunks")
corpus_matrix = vectorizer.fit_transform(chunk_texts)  # shape: (num_chunks, vocab_size)

vocab_size  = corpus_matrix.shape[1]
num_chunks  = corpus_matrix.shape[0]
sparsity    = 1 - corpus_matrix.nnz / (num_chunks * vocab_size)    # nnz: number of non-zero elements

print(f"\nCorpus matrix : {num_chunks:,} chunks  x  {vocab_size:,} vocab terms")
print(f"Sparsity      : {sparsity:.4f}  (most entries are zero — that's expected)")


# Start a new wandb run to track this script.
# run = wandb.init(
#     entity="vivekkumarkamal-indian-institute-of-technology-madras",
#     name="tfidf-vectorizer",
#     project="21f3002412-t22026",
#     config={
#         "vocab_terms": vocab_size,
#         "num_chunks": len(all_chunks),
#         "sparsity": sparsity,
#         "sublinear_tf":True,
#         "min_df":2,
#         "max_df":0.95,
#         "ngram_range":(1, 2),
#         "stop_words":"english",
#     },
# )

# run.finish()

Fitting TF-IDF vectorizer on all chunks

Corpus matrix : 38,891 chunks  x  114,318 vocab terms
Sparsity      : 0.9998  (most entries are zero — that's expected)


## Top-k chuns retrieval

**Cosine similarity** measures the angle between two vectors-> 1.0 = identical direction (most relevant), 0.0 = no shared terms.

In [94]:
def retrieve_top_k_chunks(query: str, top_k: int = 5) -> list:
    # Returns: List of chunk text strings ordered by descending relevance.
    
    query_vec = vectorizer.transform([query])         # shape: (1, vocab_size)

    # cosine_similarity returns a (1, num_chunks) matrix; .flatten() → 1-D array
    scores = cosine_similarity(query_vec, corpus_matrix).flatten()
    
    # np.argpartition: smaller scores on left, bigger scores on right of position (-top_k)
    # np.argsort: sort the top_k, and then reverse the list
    
    top_indices = np.argpartition(scores, -top_k)[-top_k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

    return [all_chunks[i]["text"] for i in top_indices]


# Quick sanity check
demo_query = "What is the Ozma Problem?"
retrieved  = retrieve_top_k_chunks(demo_query, top_k=3)

print(f"Query: '{demo_query}'\n")
for rank, chunk in enumerate(retrieved, 1):
    print(f"── Rank #{rank} ──")
    print(chunk)
    print()

Query: 'What is the Ozma Problem?'

── Rank #1 ──
=== The Ozma Problem ===
The 18th chapter, "The Ozma Problem", poses a problem that Gardner claims would arise if Earth should ever enter into communication with life on another planet through Project Ozma. This is the problem of how to communicate the meaning of left and right, where the two communicants are conditionally not allowed to view any one object in common.


── Rank #2 ──
Project Ozma, a pioneering experiment in the 1960s in the search for extraterrestrial intelligence
The Ozma problem, a philosophical problem posed in The Ambidextrous Universe by Martin Gardner
Ozuma (オズマ; stylized as OZMA), a 2012 Japanese anime television series
Frances Ozma Baum Mantele, L. Frank Baum's first granddaughter.

── Rank #3 ──
Ozma

---

Ozma may refer to:

== Fictional characters ==
Princess Ozma, ruler of the fictional land of Oz
Ozma Lee, in Macross Frontier
Ozma, a superboss in Final Fantasy IX and Final Fantasy XIV
Ozma, the Ra-seru of t

## Scoring & Prediction

Here: Matchin "prompt + option" text with the wikipedia context. 
Score based on how similar the combination is to the context.

In [95]:
OPTION_COLS = ["A", "B", "C", "D", "E"]

def rag_predict_row(row: dict, top_k_retrieval: int = 5) -> list:

    # Steps:
      # 1. Retrieve top_k Wikipedia chunks relevant to the question
      # 2. Concatenate them into a context string
      # 3. Score each option: cosine_sim(context, prompt+option)
      # 4. Return top-3 option labels in descending score order
      
    prompt = str(row["prompt"])

    # Retrieve
    retrieved_chunks = retrieve_top_k_chunks(prompt, top_k=top_k_retrieval)

    if not retrieved_chunks:
        return ["B", "C", "A"]

    # Context 
    context = " ".join(retrieved_chunks)  # Join retrieved chunks into one text blob
    context_vec = vectorizer.transform([context])   # shape: (1, vocab_size)

    # Score Each Option
    scores = []
    for opt in OPTION_COLS:
        hypothesis     = prompt + " " + str(row[opt])
        hypothesis_vec = vectorizer.transform([hypothesis])
        sim = cosine_similarity(context_vec, hypothesis_vec)[0][0]
        scores.append(sim)

    # Rank and Return Top-3
    scores_arr     = np.array(scores)
    sorted_indices = np.argsort(-scores_arr, kind="stable")  # descending
    return [OPTION_COLS[i] for i in sorted_indices[:3]]


# Demo
sample = train_df.iloc[233].to_dict()
pred   = rag_predict_row(sample, top_k_retrieval=5)

print("── RAG Demo ──")
print(f"Prompt     : {sample['prompt'][:90]}...")
print(f"Prediction : {pred}")
print(f"Correct    : {sample['answer']}")

── RAG Demo ──
Prompt     : Determine the correct option: What is the relationship between the complete electromagneti...
Prediction : ['D', 'A', 'B']
Correct    : D


## 8. Evaluate on Training Set (MAP@3)

### What is MAP@3?
The competition scores submissions with **Mean Average Precision at 3 (MAP@3)**.

For a single question with correct answer `X` and predictions `[P1, P2, P3]`:

| Prediction list | Score |
|----------------|-------|
| `[X, ?, ?]` | 1.0 (correct first) |
| `[?, X, ?]` | 0.5 (correct second) |
| `[?, ?, X]` | 0.33 (correct third) |
| `[?, ?, ?]` | 0.0 (not in top-3) |

**MAP@3** = average of these scores over all questions in the test set.

In [96]:
# Avg. Precision at K for a single question
def apk(actual: str, predicted: list, k: int = 3) -> float:
    # Returns: float in range [0, 1]

    predicted = predicted[:k]
    score = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            score += 1 / (i + 1.0)   # precision at position i+1
    return score



# Mean Average Precision at K over all questions.
def mapk(actuals: list, all_preds: list, k: int = 3) -> float:
   
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, all_preds)]))

# quick demo
apk("A", ['D', 'A', 'B', 'C', 'E'])

0.5

In [97]:
N_EVAL = len(train_df)

eval_df     = train_df
predictions = []
actuals     = []

for _, row in tqdm(eval_df.iterrows(), total=N_EVAL, desc=f"Eval ({N_EVAL} rows)"):
    predictions.append(rag_predict_row(row.to_dict(), top_k_retrieval=5))
    actuals.append(row["answer"])

score = mapk(actuals, predictions, k=3)
print(f"\nMAP@3 on first {N_EVAL} training rows: {score:.5f}")

# Old corpus
# MAP@3 on first 2000 training rows: 0.54358


Eval (2000 rows): 100%|██████████| 2000/2000 [00:12<00:00, 163.43it/s]



MAP@3 on first 2000 training rows: 0.52542


In [98]:
# Per-label breakdown — reveals if the model is biased toward certain options
per_label = defaultdict(list)
for actual, pred in zip(actuals, predictions):
    per_label[actual].append(apk(actual, pred, k=3))

print(f"{'Label':<8} {'Count':<8} {'Mean AP@3'}")
print("-" * 30)
for label in sorted(per_label):
    vals = per_label[label]
    print(f"{label:<8} {len(vals):<8} {np.mean(vals):.4f}")

Label    Count    Mean AP@3
------------------------------
A        369      0.5565
B        490      0.6259
C        459      0.4586
D        358      0.4846
E        324      0.4779


## LLM-Based Prediction

Instead of scoring options purely by cosine similarity, we hand the retrieved
context, the question, and the five options to an instruction-tuned LLM.

Three choices here matter a lot for MAP@3:

**1. Log-likelihood scoring, not generation.**
We do *not* ask the model to write `"C, A, D"` and then scrape letters out of the
reply. Free-form generation gives a confident rank-1 but near-noise ranks 2 and 3 —
and since `MAP@3 = p₁ + 0.5·p₂ + 0.33·p₃`, those lower ranks are worth real points.
Instead we run a **single forward pass** and compare the logits of the five letter
tokens at the answer position. That yields a calibrated ranking of all five options,
removes the parse-failure fallback (which quietly biased results toward A and B),
and is far faster than autoregressive decoding — which is what makes a 7B model
affordable inside a Kaggle session.

**2. Per-option retrieval, vectorised.**
The answer-bearing chunk usually shares wording with the *correct option*, not the
question. So we build six queries (bare prompt, plus `prompt + option` for each of
A–E) and score every chunk against all six in **one** `cosine_similarity` call,
keeping each chunk's best score. Calling `retrieve_top_k_chunks` six times instead
scans the 53K-chunk matrix six times over.

**3. Template stripping.**
Every prompt carries boilerplate ("Pick the best possible answer:", "Determine the
correct option:"). It appears in *every* question, so it contributes no retrieval
signal and only dilutes the query vector. We strip it for retrieval while keeping
the original prompt text for the LLM.

In [100]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model choice ─────────────────────────────────────────────────────────────
# 1.5B is the CPU-friendly default. On a Kaggle GPU session move up — model
# capacity is the single biggest lever on science MCQ:
#   "Qwen/Qwen2.5-7B-Instruct"    fp16 ~15 GB; device_map shards it across T4 x2
#   "Qwen/Qwen2.5-14B-Instruct"   needs LOAD_IN_4BIT = True
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LOAD_IN_4BIT   = False      # requires bitsandbytes; cuts a 7B to roughly 5 GB

llm_device = "cuda" if torch.cuda.is_available() else "cpu"

# T4 is Turing — good fp16, no real bf16 — so stay on float16 for GPU.
load_kwargs = {"dtype": torch.float16 if llm_device == "cuda" else torch.float32}
if llm_device == "cuda":
    load_kwargs["device_map"] = "auto"
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

print(f"Loading LLM: {LLM_MODEL_NAME} on {llm_device}")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, **load_kwargs)
if llm_device == "cpu":
    llm_model = llm_model.to(llm_device)   # device_map already placed it on GPU
llm_model.eval()


def _letter_token_ids(tok, letters):
    # "A" and " A" tokenize to different ids; keep both and score whichever
    # the model would actually emit at the answer position.
    ids = {}
    for letter in letters:
        variants = set()
        for surface in (letter, f" {letter}"):
            enc = tok.encode(surface, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[letter] = sorted(variants)
    return ids


LETTER_TOKEN_IDS = _letter_token_ids(llm_tokenizer, OPTION_COLS)


# ── Template stripping (retrieval queries only) ──────────────────────────────
# This boilerplate appears in every prompt, so it carries no retrieval signal
# and just dilutes the query vector. Same patterns as extraction.py.
TEMPLATE_PREFIX = re.compile(
    r"^(?:pick the best possible answer|select the most accurate option|"
    r"identify the correct statement|choose the correct answer|"
    r"determine the correct option|which of the following is correct)[:?,.]?\s*",
    flags=re.IGNORECASE,
)
TEMPLATE_SUFFIX = re.compile(
    r"\s*(?:among the listed options|from the following choices|"
    r"based on the given context|carefully)[:?,.]?$",
    flags=re.IGNORECASE,
)


def strip_template(text: str) -> str:
    return TEMPLATE_SUFFIX.sub("", TEMPLATE_PREFIX.sub("", text.strip())).strip()


def build_context(row: dict, top_k_retrieval: int = 5, max_context_chars: int = 4000) -> str:
    # Six queries at once: the bare prompt plus prompt+option for each of A-E.
    # The answer-bearing chunk often shares wording with the correct option
    # rather than with the question, so option-aware queries surface passages a
    # prompt-only query misses. Scoring all six in a single cosine_similarity
    # call avoids six separate scans of the 53K-chunk matrix.
    query_prompt = strip_template(str(row["prompt"]))
    queries = [query_prompt] + [f"{query_prompt} {row[opt]}" for opt in OPTION_COLS]

    query_vecs = vectorizer.transform(queries)           # (6, vocab_size)
    sims = cosine_similarity(query_vecs, corpus_matrix)  # (6, num_chunks)
    best = sims.max(axis=0)                              # each chunk's best score

    n_keep = min(top_k_retrieval * len(queries), len(best))
    top_idx = np.argpartition(best, -n_keep)[-n_keep:]
    top_idx = top_idx[np.argsort(best[top_idx])[::-1]]

    seen, ordered = set(), []
    for i in top_idx:
        text = all_chunks[i]["text"]
        if text not in seen:
            seen.add(text)
            ordered.append(text)

    return " ".join(ordered)[:max_context_chars]


@torch.no_grad()
def llm_predict_row(row: dict, top_k_retrieval: int = 5, max_context_chars: int = 4000) -> list:
    # Rank all five options by the model's log-likelihood of each answer letter.
    #
    # One forward pass, no decoding: read the next-token logits at the answer
    # position and compare the five letter tokens. This gives a calibrated
    # ordering for ranks 2 and 3 (which MAP@3 pays for) instead of whatever
    # order the model happened to emit, and needs no parse fallback.
    context = build_context(row, top_k_retrieval, max_context_chars)
    options_block = "\n".join(f"{opt}. {row[opt]}" for opt in OPTION_COLS)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert at answering multiple-choice questions using the "
                "provided context. Reply with a single letter: A, B, C, D or E."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {row['prompt']}\n\n"
                f"Options:\n{options_block}"
            ),
        },
    ]

    # tokenize=False so we can append the answer cue ourselves — the letter we
    # want to score is then exactly the next token the model would produce.
    text = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    ) + "Answer:"

    inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)
    logits = llm_model(**inputs).logits[0, -1].float()

    scores = np.array([
        max(logits[tid].item() for tid in LETTER_TOKEN_IDS[opt])
        for opt in OPTION_COLS
    ])
    return [OPTION_COLS[i] for i in np.argsort(-scores)[:3]]

Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cpu


Loading weights: 100%|██████████| 338/338 [00:06<00:00, 48.45it/s]


In [102]:

# Demo — same row as the TF-IDF demo above, for a side-by-side comparison
sample = train_df.iloc[0].to_dict()
pred = llm_predict_row(sample, top_k_retrieval=5)

print("── LLM RAG Demo ──")
print(f"Prompt     : {sample['prompt'][:90]}...")
print(f"Prediction : {pred}")
print(f"Correct    : {sample['answer']}")

── LLM RAG Demo ──
Prompt     : Pick the best possible answer: What is Martin Heidegger's view on the relationship between...
Prediction : ['B', 'A', 'C']
Correct    : B


In [104]:
# LLM eval. Sample while iterating — a full 2000-row pass is slow on CPU.
# Set N_EVAL = len(train_df) for the number you actually report.
N_EVAL = 200

# Sample randomly rather than taking the first N: train.csv may be grouped by
# topic, which would make a head() slice unrepresentative.
eval_df = (train_df if N_EVAL >= len(train_df)
           else train_df.sample(n=N_EVAL, random_state=0))

predictions = []
actuals     = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"LLM eval ({len(eval_df)} rows)"):
    predictions.append(llm_predict_row(row.to_dict(), top_k_retrieval=5))
    actuals.append(row["answer"])

score = mapk(actuals, predictions, k=3)
print(f"\nMAP@3 over {len(eval_df)} training rows: {score:.5f}")

# Per-label breakdown — a heavy skew here means the letter scoring is biased.
per_label = defaultdict(list)
for actual, pred in zip(actuals, predictions):
    per_label[actual].append(apk(actual, pred, k=3))

print(f"\n{'Label':<8} {'Count':<8} {'Mean AP@3'}")
print("-" * 30)
for label in sorted(per_label):
    vals = per_label[label]
    print(f"{label:<8} {len(vals):<8} {np.mean(vals):.4f}")

LLM eval (200 rows): 100%|██████████| 200/200 [13:16<00:00,  3.98s/it]



MAP@3 over 200 training rows: 0.90167

Label    Count    Mean AP@3
------------------------------
A        31       0.9194
B        41       0.9350
C        55       0.9182
D        37       0.9459
E        36       0.7778


## 9. Generate and Save the Submission File

We run RAG inference on the full test set and write `submission.csv` to `/kaggle/working/`.

Kaggle automatically makes files in `/kaggle/working/` available for download and submission.

Required format:
```
ID,Prediction
1,A B C
2,C A D
```

In [ ]:
# Switch this to rag_predict_row to regenerate the TF-IDF-only baseline.
PREDICT_FN = llm_predict_row

print(f"Running {PREDICT_FN.__name__} on {len(test_df):,} test rows...")

rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test inference"):
    pred = PREDICT_FN(row.to_dict(), top_k_retrieval=5)
    rows.append({
        "ID"         : row["id"],
        "Prediction" : " ".join(pred)   # e.g. 'B C A'
    })

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSubmission saved → {OUTPUT_PATH}")
print(f"Shape: {submission_df.shape}")

# Sanity check: a heavily skewed first-choice distribution means something broke.
print("\nFirst-choice distribution:")
print(submission_df["Prediction"].str.split().str[0].value_counts().to_string())

submission_df.head(10)

## 10. Bonus — Dense Retrieval with Sentence-Transformers (GPU)

TF-IDF only matches on **exact shared words**. A question about
*"Einstein's 1905 paper on light quanta"* may miss chunks about the *"photoelectric effect"*
because they share few surface words.

**Dense retrieval** uses a pre-trained neural model to embed text into a
384-dimensional semantic space — similar *meanings* get similar vectors.

### Kaggle GPU Setup
- Go to **Session options → Accelerator → GPU T4 x2** (free on Kaggle)
- `sentence-transformers` is pre-installed on Kaggle GPU images
- Embedding ~47K chunks takes **≈90 s on GPU** vs. ≈10 min on CPU

### Save Embeddings to Avoid Recomputing
Embeddings are saved to `/kaggle/working/chunk_embeddings.npy` so you can
load them in a follow-up session without re-running the model.

In [ ]:
# ── Dense Retrieval Toggle ─────────────────────────────────────────────────────
# Set to True when running on a Kaggle GPU session.
# sentence-transformers is pre-installed on Kaggle; no !pip needed.
DENSE_ENABLED = True

if DENSE_ENABLED:
    import torch
    from sentence_transformers import SentenceTransformer

    # Detect GPU availability (Kaggle GPU sessions have CUDA)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}") 

    # ── Load Model ────────────────────────────────────────────────────────────
    # all-MiniLM-L6-v2: 22 MB, 384-dim embeddings, fast and accurate.
    # Better alternatives (larger but slower):
    #   'all-mpnet-base-v2'          (768-dim, highest quality)
    #   'multi-qa-MiniLM-L6-cos-v1'  (optimised for Q&A retrieval)
    MODEL_NAME = "all-mpnet-base-v2"
    print(f"Loading model: {MODEL_NAME}")
    sbert = SentenceTransformer(MODEL_NAME, device=device)

    # ── Embed Corpus (with disk caching) ─────────────────────────────────────
    # Embeddings are saved to /kaggle/working/ so they survive across sessions.
    EMB_PATH = os.path.join(OUTPUT_DIR, "chunk_embeddings.npy")

    if os.path.exists(EMB_PATH):
        print(f"Loading pre-computed embeddings from {EMB_PATH}")
        chunk_embs = np.load(EMB_PATH)
    else:
        print(f"Embedding {len(chunk_texts):,} chunks on {device}...")
        chunk_embs = sbert.encode(
            chunk_texts,
            batch_size=256,              # increase if you have a large GPU
            show_progress_bar=True,
            normalize_embeddings=True,   # L2-normalise → cosine sim = dot product
            device=device,
        )
        np.save(EMB_PATH, chunk_embs)
        print(f"Embeddings saved → {EMB_PATH}")

    print(f"Embedding matrix shape: {chunk_embs.shape}")

    def dense_retrieve_top_k(query: str, top_k: int = 5) -> list:
        """
        Dense retrieval: encode the query with the same SBERT model and
        find the top_k nearest chunk embeddings via dot product
        (equivalent to cosine similarity because both sides are L2-normalised).
        """
        q_emb  = sbert.encode([query], normalize_embeddings=True, device=device)
        scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)

        if top_k >= len(scores):
            top_idx = np.argsort(scores)[::-1]
        else:
            top_idx = np.argpartition(scores, -top_k)[-top_k:]
            top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        return [all_chunks[i]["text"] for i in top_idx]

    # ── Quick demo ────────────────────────────────────────────────────────────
    print("\n── Dense Retrieval Demo ──")
    for rank, chunk in enumerate(dense_retrieve_top_k("Martin Heidegger's view", top_k=2), 1):
        print(f"Rank #{rank}: {chunk[:220]}...\n")

    print("To use dense retrieval everywhere, replace retrieve_top_k_chunks")
    print("with dense_retrieve_top_k inside rag_predict_row().")

else:
    print("Dense retrieval is OFF.")
    print("Set DENSE_ENABLED = True and enable a Kaggle GPU session to run this section.")

Using device: cpu
Loading model: all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31487.34it/s]


Embedding 37,727 chunks on cpu...


Batches: 100%|██████████| 148/148 [11:04:55<00:00, 269.56s/it]  


Embeddings saved → /Users/vivekkumarkamal/Projects/Coding/DL_GenAI_May_2026/RAG/chunk_embeddings.npy
Embedding matrix shape: (37727, 768)

── Dense Retrieval Demo ──


/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_62641/1503035407.py:51: RuntimeWarning: divide by zero encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_62641/1503035407.py:51: RuntimeWarning: overflow encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)
/var/folders/zy/cysncwk54glf7vf4zykldw040000gn/T/ipykernel_62641/1503035407.py:51: RuntimeWarning: invalid value encountered in matmul
  scores = (q_emb @ chunk_embs.T).flatten()   # (1, 384) @ (384, N) → (N,)


Rank #1: A number of Heidegger scholars are interviewed, including Hubert Dreyfus, Mark Wrathall, Albert Borgmann, John Haugeland, and Taylor Carman.

== See also ==

== Notes ==

== Bibliography ==

=== References ===

=== Works...

Rank #2: We take pleasure and enjoy ourselves as they take pleasure; we read, see and judge about literature and art as they see and judge; likewise we shrink back from the 'great mass' as they shrink back; we find 'shocking' wha...

To use dense retrieval everywhere, replace retrieve_top_k_chunks
with dense_retrieve_top_k inside rag_predict_row().


## Summary

### What We Built

| Step | Component | Notes |
|------|-----------|-------|
| Path setup | Kaggle-aware (`/kaggle/input/`, `/kaggle/working/`) | Edit slugs in Section 1 |
| Corpus loading | 587 Wikipedia `.md` files → 47K chunks | Sliding window, 500 chars |
| TF-IDF index | Sparse (chunks × vocab) matrix | bigrams, `min_df=2` |
| Retrieval | `retrieve_top_k_chunks()` | `cosine_similarity` + `argpartition` |
| Ranking | `rag_predict_row()` | scores all 5 options vs. context |
| Evaluation | MAP@3 via `apk()` / `mapk()` | per-label breakdown included |
| Submission | `submission.csv` → `/kaggle/working/` | Competition-ready format |
| Dense (bonus) | `sentence-transformers` SBERT | Needs Kaggle GPU T4 |

### Next Steps to Improve Score
1. **Enable GPU + Dense Retrieval** (Section 10) — `all-MiniLM-L6-v2` captures semantic similarity
2. **Bigger top-K** — try `top_k_retrieval=10` or `20` for hard topics
3. **Query expansion** — strip template prefixes ("Pick the best possible answer:") before retrieval
4. **Hybrid scoring** — combine TF-IDF score + dense score for best of both
5. **Cross-encoder reranking** — use a cross-encoder on top-20 to rerank to top-5